In [ ]:
!pip install jax-fem meshio gmsh --quiet
# JAX'ın GPU sürümünü Colab'da doğrula
import jax
print(f"Cihaz: {jax.devices()}") # 'gpu' görmelisin

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.2/166.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 16.5 MB/s eta 0:00:00
Cihaz: [CpuDevice(id=0)]


In [ ]:
!apt-get install -y libglu1-mesa libxcursor1 libxinerama1 libxft2 libxrender1

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libxcursor1 is already the newest version (1:1.2.0-2build4).
libxcursor1 set to manually installed.
libxft2 is already the newest version (2.3.4-1).
libxft2 set to manually installed.
libxinerama1 is already the newest version (2:1.1.4-3).
libxinerama1 set to manually installed.
libxrender1 is already the newest version (1:0.9.10-1build4).
The following NEW packages will be installed:
  libglu1-mesa
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 145 kB of archives.
After this operation, 367 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglu1-mesa amd64 9.0.2-1 [145 kB]
Fetched 145 kB in 0s (312 kB/s)
Selecting previously unselected package libglu1-mesa:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libglu1-mesa_9.0.2-1_amd64.deb ...
Unpacking libglu1-mes

In [ ]:
!pip install gmsh

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap
import gmsh
import numpy as np

print(f"Aktif Cihaz: {jax.devices()}") # 'gpu' görmelisin

Aktif Cihaz: [CpuDevice(id=0)]


In [ ]:
!pip install scikit-fem[all]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 8.4 MB/s eta 0:00:00


In [ ]:
import os, gmsh, h5py, numpy as np, matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
from skfem import *
from skfem import utils
from skfem.models.poisson import laplace, mass # Önceki hatayı önleyen import
from multiprocessing import Pool, cpu_count
from tqdm.notebook import tqdm

# Ayarlar
H5_FILENAME = "rf_cavity_1000_dataset.h5"
PLOT_DIR = "dataset_plots"
N_TOTAL = 1000
N_PLOT = 100
os.makedirs(PLOT_DIR, exist_ok=True)

# ============================================================================
# 1. OPTİMİZE EDİLMİŞ GEOMETRİ & MESH MOTORU
# ============================================================================
def generate_sample_data(s_id):
    try:
        gmsh.initialize()
        gmsh.model.add(f"rf_{s_id}")
        np.random.seed(s_id * 13)
        L, cx, cy = 0.1, 0.05, 0.05

        # Geometrik Çeşitlilik: Keskin köşeler ve kaotik bloblar
        method = np.random.choice(['sharp', 'smooth'])
        if method == 'sharp':
            n_pts = np.random.randint(7, 13)
            angles = np.sort(np.random.uniform(0, 2*np.pi, n_pts))
            r = np.random.uniform(0.02, 0.046, n_pts)
            pts_c = [(cx + ri*np.cos(ai), cy + ri*np.sin(ai)) for ri, ai in zip(r, angles)]
        else:
            t = np.linspace(0, 2*np.pi, 100, endpoint=False)
            r = 0.035 + sum(np.random.uniform(-0.008, 0.008) * np.cos(k*t + np.random.uniform(0, 2*np.pi)) for k in range(2, 8))
            pts_c = [(cx + ri*np.cos(ti), cy + ri*np.sin(ti)) for ri, ti in zip(r, t)]

        pts = [gmsh.model.occ.addPoint(p[0], p[1], 0) for p in pts_c]
        lines = [gmsh.model.occ.addLine(pts[i], pts[(i+1)%len(pts)]) for i in range(len(pts))]
        gmsh.model.occ.addPlaneSurface([gmsh.model.occ.addCurveLoop(lines)])
        gmsh.model.occ.synchronize()

        # Adaptif Mesh: Sınırda yoğun, merkezde dengeli
        gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
        gmsh.model.mesh.field.add("Distance", 1)
        gmsh.model.mesh.field.setNumbers(1, "CurvesList", lines)
        gmsh.model.mesh.field.add("Threshold", 2)
        gmsh.model.mesh.field.setNumber(2, "InField", 1)
        gmsh.model.mesh.field.setNumber(2, "SizeMin", 0.0012)
        gmsh.model.mesh.field.setNumber(2, "SizeMax", 0.005)
        gmsh.model.mesh.field.setNumber(2, "DistMin", 0.002)
        gmsh.model.mesh.field.setNumber(2, "DistMax", 0.03)
        gmsh.model.mesh.field.setAsBackgroundMesh(2)

        gmsh.model.mesh.generate(2)
        _, coords, _ = gmsh.model.mesh.getNodes()
        _, _, conns = gmsh.model.mesh.getElements(2)
        nodes = np.ascontiguousarray(coords.reshape(-1, 3)[:, :2])
        elements = np.ascontiguousarray((conns[0].reshape(-1, 3) - 1).astype(np.int32))
        gmsh.finalize()

        # Physics Solver (P2 Precision)
        m = MeshTri(nodes.T, elements.T)
        basis = Basis(m, ElementTriP2())
        K, M = laplace.assemble(basis), mass.assemble(basis)
        D = basis.get_dofs(facets=m.boundary_facets())
        Kc, Mc, xc, Ic = utils.condense(K, M, D=D, expand=True)
        vals, vecs = utils.solve_eigen(Kc, Mc, x=xc, I=Ic, k=3, sigma=500.0)

        freqs = (299792458 * np.sqrt(np.abs(vals.real))) / (2 * np.pi) / 1e9

        return {
            'id': s_id, 'nodes': nodes, 'elements': elements,
            'freqs': freqs.real, 'vecs': vecs.real, 'n_nodes': len(nodes)
        }
    except Exception as e:
        return None

# ============================================================================
# 2. GÖRSELLEŞTİRME YARDIMCISI
# ============================================================================
def save_sample_plot(data, save_path):
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    nodes, elements = data['nodes'], data['elements']
    triang = Triangulation(nodes[:, 0], nodes[:, 1], elements)

    # Mesh Panel
    axes[0].triplot(triang, color='gray', linewidth=0.15, alpha=0.5)
    axes[0].set_title(f"ID {data['id']}: {len(elements)} Elements")

    for i in range(3):
        # P2'den ilk n_nodes kadarını görselleştirme için al
        mode_v = data['vecs'][:data['n_nodes'], i]
        mode_norm = mode_v / np.max(np.abs(mode_v))
        axes[i+1].tripcolor(triang, mode_norm, shading='gouraud', cmap='RdBu_r', vmin=-1, vmax=1)
        axes[i+1].set_title(f"Mode {i+1}: {data['freqs'][i]:.4f} GHz")

    for ax in axes: ax.set_aspect('equal'); ax.axis('off')
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()

# ============================================================================
# 3. ANA DÖNGÜ & HDF5 PAKETLEME
# ============================================================================
if __name__ == '__main__':
    print(f"🚀 {cpu_count()} çekirdek ile 1000 örnek üretiliyor...")

    with h5py.File(H5_FILENAME, "w") as f_h5:
        with Pool(cpu_count()) as pool:
            # Örnekleri chunklar halinde işle (bellek yönetimi için)
            for i in tqdm(range(0, N_TOTAL, 50), desc="Toplu Üretim"):
                chunk_range = range(i, min(i + 50, N_TOTAL))
                chunk_results = pool.map(generate_sample_data, chunk_range)

                for res in chunk_results:
                    if res is None: continue

                    # H5 Kayıt (Gzip sıkıştırma ile)
                    grp = f_h5.create_group(f"sample_{res['id']:04d}")
                    grp.create_dataset("nodes", data=res['nodes'], compression="gzip", compression_opts=4)
                    grp.create_dataset("elements", data=res['elements'], compression="gzip", compression_opts=4)
                    grp.create_dataset("freqs", data=res['freqs'])
                    grp.create_dataset("vecs", data=res['vecs'], compression="gzip", compression_opts=4)

                    # İlk 100 tanesini görsel olarak kaydet
                    if res['id'] < N_PLOT:
                        plot_path = os.path.join(PLOT_DIR, f"sample_{res['id']:03d}.png")
                        save_sample_plot(res, plot_path)

    print(f"\n✅ Tamamlandı! \n📦 Dosya: {H5_FILENAME} \n🖼️ Görseller: {PLOT_DIR}/")

🚀 2 çekirdek ile 1000 örnek üretiliyor...


/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Toplu Üretim:   0%|          | 0/20 [00:00<?, ?it/s]

/usr/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/local/lib/python3.12/dist-packages/skfem/utils.py:216: ComplexWarning: Casting complex values to real discards the imaginary part
  y[I] = X
/usr/local/lib/python3.12/dist-packages/skfem/utils.py:216: ComplexWarning: Casting complex values to real discards the imaginary part
  y[I] = X


In [ ]:
from google.colab import files
import os

files.download("rf_cavity_1000_dataset.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>